# Mamba3 MIMO 40M — English smoke

This notebook keeps the MIMO architecture gate and uses the shared English FineWeb-Edu smoke corpus. On a T4, a TileLang import failure is a hardware/software gate failure, not a dataset failure.

In [ ]:
from pathlib import Path
import subprocess, sys, torch
REPO=Path('/workspace/murmur-science')
if not REPO.exists(): subprocess.run(['git','clone','--branch','codex/mamba3-mimo-smoke','https://github.com/orkrs/murmur-science.git',str(REPO)],check=True)
%cd /workspace/murmur-science
sys.path.insert(0,str(Path.cwd()/'src'))
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU is required')
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
%pip install -q datasets sentencepiece pyarrow pandas einops ninja
%pip install -q --upgrade 'tilelang==0.1.8' 'apache-tvm-ffi<=0.1.12' 'quack-kernels>=0.3.4' 'triton>=3.5.0'
import os; os.environ['MAMBA_FORCE_BUILD']='TRUE'
%pip install -q --no-cache-dir --no-deps --force-reinstall --no-build-isolation git+https://github.com/state-spaces/mamba.git@main

In [ ]:
from murmur.config import load_run_config
config=load_run_config(Path('configs/smoke_mamba3_mimo.toml'))
assert config.model.mixer=='mamba3_mimo'
from mamba_ssm.modules.mamba3 import Mamba3
gate=Mamba3(d_model=128,d_state=64,headdim=64,is_mimo=True,mimo_rank=2,chunk_size=32,dtype=torch.bfloat16).cuda().train()
x=torch.randn(1,512,128,device='cuda',dtype=torch.bfloat16,requires_grad=True)
loss=gate(x).float().square().mean(); loss.backward()
print('Verified Mamba3 MIMO forward/backward gate')

In [ ]:
subprocess.run([sys.executable,'scripts/build_hf_mix.py','--profile','english_smoke','--output','artifacts/english_smoke_corpus','--max-tokens','2400000'],check=True)
!python scripts/train_tokenizer.py --corpus artifacts/english_smoke_corpus/corpus.txt --output artifacts/english_smoke_tokenizer.model --vocab-size 32000
!python scripts/prepare_data.py --config configs/smoke_mamba3_mimo.toml --tokenizer artifacts/english_smoke_tokenizer.model --train-input artifacts/english_smoke_corpus/train.jsonl --val-input artifacts/english_smoke_corpus/val.jsonl --output artifacts/english_smoke_data

In [ ]:
template=Path('configs/smoke_mamba3_mimo.toml').read_text()
Path('configs/smoke_mamba3_mimo_session.toml').write_text(template.replace('artifacts/data/train.bin','artifacts/english_smoke_data/train.bin').replace('artifacts/data/val.bin','artifacts/english_smoke_data/val.bin'))
run_dir=Path('artifacts/runs/mimo_english_smoke')
subprocess.run([sys.executable,'scripts/train.py','--config','configs/smoke_mamba3_mimo_session.toml','--run-dir',str(run_dir),'--device','cuda'],check=True)
assert (run_dir/'checkpoints'/'last'/'COMPLETED').exists()
print('MIMO English smoke checkpoint ready')